Import the necessary libraries for data manipulation

In [1]:
import pandas as pd

# script B-03 - Cultivation - Import Cultivation

Parent Company Extraction from License Data

In [2]:
parent_df = pd.read_csv(
    "Data/Cal Poly/Cannabis Market Intelligence Platform Report - Licenses - 2025-07-03.csv",
    dtype=str,
    keep_default_na=False,
)
parent_df = parent_df.rename(
    columns={
        "Company ID": "companyid",
        "Country": "county",
        "State License ID": "statelicenseid",
    }
)

parent_df["multi_owner"] = 0
parent_df["multi_owner"] = parent_df["companyid"].str.find(";")
parent_df["multi_owner"] = parent_df["multi_owner"].replace(-1, 0)
parent_df["primary_company"] = parent_df.apply(
    lambda row: row["companyid"][: row["multi_owner"]]
    if row["multi_owner"] > 0
    else row["companyid"],
    axis=1,
)

parent_df["primary_company"] = pd.to_numeric(
    parent_df["primary_company"], errors="coerce"
)


parent_df = parent_df.rename(columns={"statelicenseid": "licenseNumber"})

parent_df = parent_df[["licenseNumber", "primary_company"]]

parent_df = parent_df.drop_duplicates()
parent_df = parent_df[
    parent_df["licenseNumber"].notna() & (parent_df["licenseNumber"] != "")
]

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Cal Poly/Cannabis Market Intelligence Platform Report - Licenses - 2025-07-03.csv'

Cultivation Data Import, Filtering, and Type Assignment

In [3]:
cultivation_df = pd.read_excel(
    "Data/Cal Poly/Working Cultivation Canopy June 2025.xlsx",
    sheet_name="License data canopy",
    engine="openpyxl",
    dtype=str,
    keep_default_na=False,
)

cultivation_df = cultivation_df[cultivation_df["licenseStatus"] == "Active"]

cultivation_df = cultivation_df[cultivation_df["LargeDate"].isna()]

mapping = {
    "Large Indoor": "Indoor",
    "Large Mixed-Light Tier 1": "Mixed_Light",
    "Large Mixed-Light Tier 2": "Mixed_Light",
    "Large Outdoor": "Outdoor",
    "Medium Indoor": "Indoor",
    "Medium Mixed-Light Tier 1": "Mixed_Light",
    "Medium Mixed-Light Tier 2": "Mixed_Light",
    "Medium Outdoor": "Outdoor",
    "Small Indoor": "Indoor",
    "Small Mixed-Light Tier 1": "Mixed_Light",
    "Small Mixed-Light Tier 2": "Mixed_Light",
    "Small Outdoor": "Outdoor",
    "Specialty Cottage Indoor": "Indoor",
    "Specialty Cottage Mixed-Light Tier 1": "Mixed_Light",
    "Specialty Cottage Mixed-Light Tier 2": "Mixed_Light",
    "Specialty Cottage Outdoor": "Outdoor",
    "Specialty Indoor": "Indoor",
    "Specialty Mixed-Light Tier 1": "Mixed_Light",
    "Specialty Mixed-Light Tier 2": "Mixed_Light",
    "Specialty Outdoor": "Outdoor",
}

cultivation_df["type"] = cultivation_df["licenseType"].map(mapping).fillna("")

cultivation_df["micro_cult"] = (
    cultivation_df["activity"].astype(str).str.contains("Cultivator").astype(int)
)
cultivation_df["micro_indoor"] = (
    cultivation_df["activity"].astype(str).str.contains("Indoor").astype(int)
)

cultivation_df = cultivation_df[
    ~(
        (cultivation_df["licenseType"] == "Microbusiness")
        & (cultivation_df["micro_cult"] == 0)
    )
]

cond_outdoor = (cultivation_df["licenseType"] == "Microbusiness") & (
    cultivation_df["micro_indoor"] == 0
)
cond_indoor = (cultivation_df["licenseType"] == "Microbusiness") & (
    cultivation_df["micro_indoor"] == 1
)

cultivation_df.loc[cond_outdoor, "type"] = "Outdoor"
cultivation_df.loc[cond_indoor, "type"] = "Indoor"

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Cal Poly/Working Cultivation Canopy June 2025.xlsx'

Merge cultivation data with parent company data on licenseNumber and save it

In [ ]:
parent_df["licenseNumber"] = parent_df["licenseNumber"].astype(str)
cultivation_df["licenseNumber"] = cultivation_df["licenseNumber"].astype(str)

merged_df = cultivation_df.merge(
    parent_df, on="licenseNumber", how="inner", validate="many_to_one"
)
merged_df.to_stata("Data/Working_data/cultivation.dta", write_index=False)

# script B-04 - Cultivation- Calculate_Cultivation_Concentration

Cultivation HHI Calculation by Grow Type

In [ ]:
cultivation_df = pd.read_stata("Data/Working_data/cultivation.dta")

grow_types = ["Indoor", "Mixed_Light", "Outdoor"]

for grow_type in grow_types:
    df = cultivation_df[cultivation_df["type"] == grow_type].copy()

    # === Statewide HHI overall ===
    statewide_overall = (
        df.groupby("businessLegalName")
        .agg({"Canopy": "sum", "MaxSqFt": "sum"})
        .reset_index()
    )

    industry_Canopy = statewide_overall["Canopy"].sum()
    industry_MaxSqFt = statewide_overall["MaxSqFt"].sum()

    statewide_overall["mkt_share_Canopy"] = (
        statewide_overall["Canopy"] / industry_Canopy
    ) * 100
    statewide_overall["mkt_share2_Canopy"] = statewide_overall["mkt_share_Canopy"] ** 2

    statewide_overall["mkt_share_MaxSqFt"] = (
        statewide_overall["MaxSqFt"] / industry_MaxSqFt
    ) * 100
    statewide_overall["mkt_share2_MaxSqFt"] = (
        statewide_overall["mkt_share_MaxSqFt"] ** 2
    )

    CA_overall = pd.DataFrame(
        {
            "mkt_share2_Canopy": [statewide_overall["mkt_share2_Canopy"].sum()],
            "Canopy": [statewide_overall["Canopy"].sum()],
            "mkt_share2_MaxSqFt": [statewide_overall["mkt_share2_MaxSqFt"].sum()],
            "MaxSqFt": [statewide_overall["MaxSqFt"].sum()],
        }
    )
    CA_overall["premiseCounty"] = "CA"
    CA_overall["level"] = "Overall"

    # === Statewide HHI Parent Company ===

    parent_group = (
        df.groupby("primary_company")
        .agg({"Canopy": "sum", "MaxSqFt": "sum", "businessLegalName": "first"})
        .reset_index()
    )

    parent_business = (
        parent_group.groupby("businessLegalName")
        .agg({"Canopy": "sum", "MaxSqFt": "sum"})
        .reset_index()
    )

    industry_Canopy = parent_business["Canopy"].sum()
    industry_MaxSqFt = parent_business["MaxSqFt"].sum()

    parent_business["mkt_share_Canopy"] = (
        parent_business["Canopy"] / industry_Canopy
    ) * 100
    parent_business["mkt_share2_Canopy"] = parent_business["mkt_share_Canopy"] ** 2

    parent_business["mkt_share_MaxSqFt"] = (
        parent_business["MaxSqFt"] / industry_MaxSqFt
    ) * 100
    parent_business["mkt_share2_MaxSqFt"] = parent_business["mkt_share_MaxSqFt"] ** 2

    CA_parent = pd.DataFrame(
        {
            "mkt_share2_Canopy": [parent_business["mkt_share2_Canopy"].sum()],
            "Canopy": [parent_business["Canopy"].sum()],
            "mkt_share2_MaxSqFt": [parent_business["mkt_share2_MaxSqFt"].sum()],
            "MaxSqFt": [parent_business["MaxSqFt"].sum()],
        }
    )
    CA_parent["premiseCounty"] = "CA"
    CA_parent["level"] = "Parent Company"

    # === County-level HHI overall ===

    county_overall = (
        df.groupby(["businessLegalName", "premiseCounty"])
        .agg({"Canopy": "sum", "MaxSqFt": "sum"})
        .reset_index()
    )

    industry_sum_canopy = county_overall.groupby("premiseCounty")["Canopy"].transform(
        "sum"
    )
    industry_sum_maxsqft = county_overall.groupby("premiseCounty")["MaxSqFt"].transform(
        "sum"
    )

    county_overall["mkt_share_Canopy"] = (
        county_overall["Canopy"] / industry_sum_canopy
    ) * 100
    county_overall["mkt_share2_Canopy"] = county_overall["mkt_share_Canopy"] ** 2

    county_overall["mkt_share_MaxSqFt"] = (
        county_overall["MaxSqFt"] / industry_sum_maxsqft
    ) * 100
    county_overall["mkt_share2_MaxSqFt"] = county_overall["mkt_share_MaxSqFt"] ** 2

    county_overall_agg = (
        county_overall.groupby("premiseCounty")
        .agg(
            {
                "mkt_share2_Canopy": "sum",
                "Canopy": "sum",
                "mkt_share2_MaxSqFt": "sum",
                "MaxSqFt": "sum",
            }
        )
        .reset_index()
    )
    county_overall_agg["level"] = "Overall"

    # === County-level HHI Parent Company ===
    parent_county_group = (
        df.groupby(["primary_company", "premiseCounty"])
        .agg({"Canopy": "sum", "MaxSqFt": "sum", "businessLegalName": "first"})
        .reset_index()
    )

    parent_county_business = (
        parent_county_group.groupby(["businessLegalName", "premiseCounty"])
        .agg({"Canopy": "sum", "MaxSqFt": "sum"})
        .reset_index()
    )

    industry_sum_canopy_parent = parent_county_business.groupby("premiseCounty")[
        "Canopy"
    ].transform("sum")
    industry_sum_maxsqft_parent = parent_county_business.groupby("premiseCounty")[
        "MaxSqFt"
    ].transform("sum")

    parent_county_business["mkt_share_Canopy"] = (
        parent_county_business["Canopy"] / industry_sum_canopy_parent
    ) * 100
    parent_county_business["mkt_share2_Canopy"] = (
        parent_county_business["mkt_share_Canopy"] ** 2
    )

    parent_county_business["mkt_share_MaxSqFt"] = (
        parent_county_business["MaxSqFt"] / industry_sum_maxsqft_parent
    ) * 100
    parent_county_business["mkt_share2_MaxSqFt"] = (
        parent_county_business["mkt_share_MaxSqFt"] ** 2
    )

    county_parent_agg = (
        parent_county_business.groupby("premiseCounty")
        .agg(
            {
                "mkt_share2_Canopy": "sum",
                "Canopy": "sum",
                "mkt_share2_MaxSqFt": "sum",
                "MaxSqFt": "sum",
            }
        )
        .reset_index()
    )
    county_parent_agg["level"] = "Parent Company"

    # === Combine all results ===
    combined = pd.concat(
        [county_overall_agg, CA_overall, county_parent_agg, CA_parent],
        ignore_index=True,
        sort=False,
    )
    numeric_cols = combined.select_dtypes(include=["number"]).columns
    combined[numeric_cols] = combined[numeric_cols].round(0).astype(int)
    final_export = combined.astype(str)
    output_path = f"Data/Results/Cult_HHI__{grow_type}_test.xlsx"
    final_export.to_excel(output_path, index=False)

    print(f"Saved {output_path}")

Saved Data/Results/Cult_HHI__Indoor_test.xlsx
Saved Data/Results/Cult_HHI__Mixed_Light_test.xlsx
Saved Data/Results/Cult_HHI__Outdoor_test.xlsx


In [4]:
cultivation_df = pd.read_stata("Data/Working_data/cultivation.dta")

In [5]:
cultivation_df.columns

Index(['id', 'licenseNumber', 'licenseStatus', 'licenseTerm', 'licenseType',
       'licenseDesignation', 'issueDate', 'expirationDate',
       'licenseStatusDate', 'businessLegalName', 'businessDbaName',
       'businessOwnerName', 'businessStructure', 'activity',
       'premiseStreetAddress', 'premiseCity', 'premiseState', 'premiseCounty',
       'premiseZipCode', 'businessEmail', 'businessPhone', 'parcelNumber',
       'PremiseLatitude', 'PremiseLongitude', 'Canopy', 'LargeDate', 'MaxSqFt',
       'Utilized', 'MaxLbsOld', 'AdjLbsOld', 'MaxLbsNew', 'AdjLbsNew',
       'LbsSqft', 'LbsSqftNew', 'type', 'micro_cult', 'micro_indoor',
       'primary_company'],
      dtype='object')

In [6]:
import pandas as pd

cultivation_df = pd.read_stata("Data/Working_data/cultivation.dta")

grow_types = ["Indoor", "Mixed_Light", "Outdoor"]

results_list = []

for grow_type in grow_types:
    df = cultivation_df[cultivation_df["type"] == grow_type].copy()

    # === Statewide HHI (Overall) ===
    statewide = df.groupby("businessLegalName").agg({"Canopy":"sum","MaxSqFt":"sum"}).reset_index()
    industry_Canopy = statewide["Canopy"].sum()
    industry_MaxSqFt = statewide["MaxSqFt"].sum()
    
    statewide["mkt_share_Canopy"] = (statewide["Canopy"]/industry_Canopy)*100
    statewide["mkt_share2_Canopy"] = statewide["mkt_share_Canopy"]**2
    statewide["mkt_share_MaxSqFt"] = (statewide["MaxSqFt"]/industry_MaxSqFt)*100
    statewide["mkt_share2_MaxSqFt"] = statewide["mkt_share_MaxSqFt"]**2
    
    results_list.append({
        "grow_type": grow_type,
        "geography": "CA",
        "level": "Overall",
        "HHI_Canopy": statewide["mkt_share2_Canopy"].sum(),
        "Total_Canopy": industry_Canopy,
        "HHI_MaxSqFt": statewide["mkt_share2_MaxSqFt"].sum(),
        "Total_MaxSqFt": industry_MaxSqFt
    })
    
    # === Statewide HHI (Parent) ===
    parent = df.groupby("primary_company").agg({"Canopy":"sum","MaxSqFt":"sum"}).reset_index()
    industry_Canopy = parent["Canopy"].sum()
    industry_MaxSqFt = parent["MaxSqFt"].sum()
    
    parent["mkt_share_Canopy"] = (parent["Canopy"]/industry_Canopy)*100
    parent["mkt_share2_Canopy"] = parent["mkt_share_Canopy"]**2
    parent["mkt_share_MaxSqFt"] = (parent["MaxSqFt"]/industry_MaxSqFt)*100
    parent["mkt_share2_MaxSqFt"] = parent["mkt_share_MaxSqFt"]**2
    
    results_list.append({
        "grow_type": grow_type,
        "geography": "CA",
        "level": "Parent Company",
        "HHI_Canopy": parent["mkt_share2_Canopy"].sum(),
        "Total_Canopy": industry_Canopy,
        "HHI_MaxSqFt": parent["mkt_share2_MaxSqFt"].sum(),
        "Total_MaxSqFt": industry_MaxSqFt
    })
    
    # === County-level HHI (Overall) ===
    county = df.groupby(["premiseCounty","businessLegalName"]).agg({"Canopy":"sum","MaxSqFt":"sum"}).reset_index()
    county["industry_Canopy"] = county.groupby("premiseCounty")["Canopy"].transform("sum")
    county["industry_MaxSqFt"] = county.groupby("premiseCounty")["MaxSqFt"].transform("sum")
    
    county["mkt_share_Canopy"] = (county["Canopy"]/county["industry_Canopy"])*100
    county["mkt_share2_Canopy"] = county["mkt_share_Canopy"]**2
    county["mkt_share_MaxSqFt"] = (county["MaxSqFt"]/county["industry_MaxSqFt"])*100
    county["mkt_share2_MaxSqFt"] = county["mkt_share_MaxSqFt"]**2
    
    county_agg = county.groupby("premiseCounty").agg(
        HHI_Canopy=("mkt_share2_Canopy","sum"),
        Total_Canopy=("Canopy","sum"),
        HHI_MaxSqFt=("mkt_share2_MaxSqFt","sum"),
        Total_MaxSqFt=("MaxSqFt","sum")
    ).reset_index()
    county_agg["grow_type"] = grow_type
    county_agg["level"] = "Overall"
    
    results_list.extend(county_agg.to_dict("records"))
    
    # === County-level HHI (Parent) ===
    parent_county = df.groupby(["premiseCounty","primary_company"]).agg({"Canopy":"sum","MaxSqFt":"sum"}).reset_index()
    parent_county["industry_Canopy"] = parent_county.groupby("premiseCounty")["Canopy"].transform("sum")
    parent_county["industry_MaxSqFt"] = parent_county.groupby("premiseCounty")["MaxSqFt"].transform("sum")
    
    parent_county["mkt_share_Canopy"] = (parent_county["Canopy"]/parent_county["industry_Canopy"])*100
    parent_county["mkt_share2_Canopy"] = parent_county["mkt_share_Canopy"]**2
    parent_county["mkt_share_MaxSqFt"] = (parent_county["MaxSqFt"]/parent_county["industry_MaxSqFt"])*100
    parent_county["mkt_share2_MaxSqFt"] = parent_county["mkt_share_MaxSqFt"]**2
    
    parent_county_agg = parent_county.groupby("premiseCounty").agg(
        HHI_Canopy=("mkt_share2_Canopy","sum"),
        Total_Canopy=("Canopy","sum"),
        HHI_MaxSqFt=("mkt_share2_MaxSqFt","sum"),
        Total_MaxSqFt=("MaxSqFt","sum")
    ).reset_index()
    parent_county_agg["grow_type"] = grow_type
    parent_county_agg["level"] = "Parent Company"
    
    results_list.extend(parent_county_agg.to_dict("records"))

# Final dataframe with all HHI results
HHI_results = pd.DataFrame(results_list)

# --- Export ---
HHI_results.to_excel("Data/Results/Cult_HHI_DeepDive.xlsx", index=False)

print("Saved Data/Results/Cult_HHI_DeepDive.xlsx")



Saved Data/Results/Cult_HHI_DeepDive.xlsx


In [7]:
import pandas as pd

HHI_results = pd.DataFrame(results_list)

# === Table 1: HHI Summary ===
HHI_summary = HHI_results[[
    "grow_type", "geography", "level", "HHI_Canopy", "HHI_MaxSqFt"
]]
HHI_summary.to_csv("Data/Results/Cult_HHI_Summary.csv", index=False)

# === Table 2: Total Size vs HHI ===
Size_vs_HHI = HHI_results[[
    "grow_type", "geography", "level",
    "Total_Canopy", "HHI_Canopy",
    "Total_MaxSqFt", "HHI_MaxSqFt"
]]
Size_vs_HHI.to_csv("Data/Results/Cult_Size_vs_HHI.csv", index=False)

print("Saved Cult_HHI_Summary.csv and Cult_Size_vs_HHI.csv in Data/Results/")


Saved Cult_HHI_Summary.csv and Cult_Size_vs_HHI.csv in Data/Results/
